# 01 — 2D Multi-Channel Correlative Fusion & Pixel-Level Uncertainty

This self-contained tutorial demonstrates how **X-FUSE** fuses a high-resolution structural micrograph ($I_{\text{HR}}$: e.g., XCT, SEM-BSE, HAADF-STEM, or white-beam neutron) with a coarse multi-channel physical guide ($G_{\text{LR}}$: e.g., XRD-CT, EDS, EELS, or Bragg-edge) to produce:
1. **Super-resolved phase/element probability maps** respecting physical partial-volume fractions,
2. **Tri-part pixel-level uncertainty** (calibrated posterior entropy, online Welford TTA variance, and forward-PSF physical discrepancy),
3. **Automatic detection of structurally invisible phases** when $I_{\text{HR}}$ lacks contrast.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from x_fuse import (
    CorrelativeFuser,
    build_default_tta,
    compute_segmentation_metrics,
    ensure_spatial_divisible,
    expected_calibration_error,
    generate_correlative_phantom_2d,
    guide_to_tensor,
    jbu_guided_baseline,
    normalize_structural,
    sparsification_curve,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BACKBONE = "dinov2_vits14_reg" if DEVICE == "cuda" else "timm:vit_tiny_patch16_224"
PRETRAINED = DEVICE == "cuda"
print(f"Running on {DEVICE} with backbone={BACKBONE} (pretrained={PRETRAINED})")

## 1. Generate a Controlled 3-Phase Correlative Phantom ($8\times$ Resolution Gap)
We synthesize a 2D microstructure with Matrix (`0`), Phase A (`1`, strong structural contrast), and Phase B (`2`, moderate contrast), paired with an $8\times$ coarser, PSF-blurred, shot-noise-limited 2-channel chemical guide.

In [ ]:
phantom = generate_correlative_phantom_2d(
    size=128,
    downsample_factor=8,
    phase_b_contrast=0.35,
    hr_noise_std=0.03,
    poisson_counts=250.0,
    seed=42,
)
print("HR structural shape:", phantom.hr_image.shape)
print("Coarse 2-channel guide shape:", phantom.coarse_guide.shape)

## 2. Fit Physics-Guided `CorrelativeFuser` & Compare Against Classical JBU
`CorrelativeFuser` extracts dense ViT features once and supervises the multi-channel probe via **forward PSF downsampling** ($\mathcal{D}(\hat{P}_{\text{HR}}) \approx G_{\text{LR}}$) rather than penalizing sharp grain boundaries against blurry upsampled targets.

In [ ]:
jbu_rep = jbu_guided_baseline(phantom.hr_image, phantom.coarse_guide, mode="multiclass")

patch_sz = 14 if "14" in BACKBONE else 16
stride = 7 if patch_sz == 14 else 8
hr_t, (oh, ow) = ensure_spatial_divisible(
    normalize_structural(phantom.hr_image, device=DEVICE),
    patch_size=patch_sz,
    stride=stride,
)
guide_t = guide_to_tensor(phantom.coarse_guide, device=DEVICE, normalize=True)
fwd_tta, inv_tta = build_default_tta(shift_dists=[1, 2], pattern="Neumann", include_flips=True)

fuser = CorrelativeFuser(
    dino_name=BACKBONE,
    xrd_fuse_method="learned_gating",
    stride=stride,
    dtype=torch.float32,
    loss_fn="forward_psf_bce",
    guide_channels=2,
    interpolation_mode="jbu",
    pretrained=PRETRAINED,
    lr=8e-3,
    device=DEVICE,
)
fuser.set_guide_transforms(guide_t, fwd_tta, inv_tta)

for step in range(30):
    loss = fuser.train_fusion_step(hr_t)

report = fuser.predict_with_uncertainty(
    hr_t, guide_lr=guide_t, mode="multiclass", calibrate_to_guide=True
)
xf_mask = report.hard_mask[0, :oh, :ow].cpu().numpy()
xf_unc = report.combined_uncertainty[0, :oh, :ow].cpu().numpy()
xf_probs = report.probabilities[:, :, :oh, :ow].cpu()

print("Classical JBU mIoU:", round(compute_segmentation_metrics(jbu_rep.hard_mask[0], phantom.gt_labels)["iou"], 4))
print("X-FUSE mIoU:       ", round(compute_segmentation_metrics(xf_mask, phantom.gt_labels)["iou"], 4))
print("X-FUSE ECE:        ", round(expected_calibration_error(xf_probs, phantom.gt_labels), 4))
curve = sparsification_curve(xf_mask, phantom.gt_labels, xf_unc)
print("Sparsification AUSE:", round(float(curve["ause"]), 4))

## 3. Visualise Super-Resolved Segmentation & Tri-Part Uncertainty Maps

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8), constrained_layout=True)
axes[0, 0].imshow(phantom.hr_image, cmap="gray")
axes[0, 0].set_title("High-Res Structural $I_{HR}$")
axes[0, 1].imshow(phantom.coarse_guide[0], cmap="magma")
axes[0, 1].set_title("Coarse Guide Ch 0 ($8\\times$ LR)")
axes[0, 2].imshow(xf_mask, cmap="viridis", vmin=0, vmax=2)
axes[0, 2].set_title("X-FUSE Calibrated Segmentation")

axes[1, 0].imshow(report.aleatoric_entropy[0, :oh, :ow].cpu(), cmap="inferno")
axes[1, 0].set_title("1. Posterior Entropy (Chemical)")
axes[1, 1].imshow(report.epistemic_variance[0, :oh, :ow].cpu(), cmap="inferno")
axes[1, 1].set_title("2. TTA Epistemic Variance (Spatial)")
axes[1, 2].imshow(report.physical_residual[0, :oh, :ow].cpu(), cmap="inferno")
axes[1, 2].set_title("3. Forward PSF Residual (Physics)")
for ax in axes.ravel():
    ax.axis("off")
plt.show()